# RAG conversacional sobre múltiples PDFs

Un asistente que responde preguntas sobre cuatro PDFs y **mantiene el hilo de la conversación**,
de modo que se le puede preguntar *"¿y cuántas calorías tienen?"* sin repetir de qué hablábamos.

## Objetivos

1. **Cargar** varios PDFs desde una carpeta local con `DirectoryLoader`.
2. **Trocear** las páginas en chunks con `RecursiveCharacterTextSplitter`.
3. **Indexar** esos chunks como embeddings en un vector store (`Chroma`).
4. **Recuperar** los trozos relevantes para una pregunta (`retriever`).
5. **Reformular** preguntas que dependen del historial, para que la búsqueda funcione.
6. **Conversar** con memoria, sin perder el contexto entre turnos.
7. **Aislar sesiones** con un `session_id`, para varias conversaciones en paralelo.

## Qué construimos

```mermaid
flowchart TB
    subgraph IDX["1 · Indexación · se ejecuta una vez"]
        direction LR
        A[PDFs remotos] --> B[data/pdfs]
        B --> C["Documents<br/>8 páginas"]
        C --> D["Chunks<br/>13 trozos"]
        D --> EMB1["☁️ OpenAI · embeddings<br/>text-embedding-3-small<br/>1 llamada · 13 textos"]
        EMB1 --> E[("Chroma<br/>vector store")]
    end

    subgraph CHAT["2 · Conversación · en cada turno de una sesión"]
        direction TB
        Q[pregunta del usuario] --> R{"¿hay historial?"}
        R -->|no| S["query = la pregunta tal cual"]
        R -->|sí| LLM1["☁️ Anthropic · chat<br/>claude-haiku-4-5 · reformula<br/>contextualize_chain"]
        LLM1 --> S
        S --> EMB2["☁️ OpenAI · embeddings<br/>text-embedding-3-small<br/>1 llamada · 1 texto"]
        EMB2 --> U["retriever<br/>top k=4 en Chroma"]
        U --> LLM2["☁️ Anthropic · chat<br/>claude-haiku-4-5 · responde<br/>qa_chain"]
        LLM2 --> W[respuesta]
        W --> X[("sessions[session_id]")]
        X -->|alimenta el próximo turno| R
    end

    E -.->|busca aquí| U

    classDef openai fill:#10a37f,stroke:#0b6e55,color:#ffffff,stroke-width:2px
    classDef anthropic fill:#d97757,stroke:#a3523a,color:#ffffff,stroke-width:2px
    class EMB1,EMB2 openai
    class LLM1,LLM2 anthropic
```

En verde, **las llamadas a OpenAI** (embeddings) y en naranja, **las llamadas a Anthropic** (chat).
Anthropic no ofrece embeddings, por eso el vector store sigue en OpenAI.
Conviene ver las llamadas explícitas porque dos de ellas son fáciles de olvidar: la búsqueda en Chroma **también** consume API (hay que
convertir la pregunta a embedding antes de comparar), y el paso de reformulación añade
una llamada al modelo de chat.

| Fase | embeddings | chat | total |
|---|---|---|---|
| Indexación (una vez) | 1 · con los 13 chunks en lote | – | **1** |
| Turno sin historial | 1 | 1 | **2** |
| Turno con historial | 1 | 2 | **3** |

*(Números medidos interceptando las llamadas HTTP reales, no estimados.)*

La clave está en el rombo: **sin el paso de reformulación**, buscar literalmente
*"¿y cuántas calorías tienen?"* en el vector store no encontraría nada útil,
porque esa frase no se parece a ningún texto de los PDFs. Esa precisión extra
cuesta una llamada más por turno.

## Cómo transcurre un turno

El diagrama de arriba muestra las piezas; este muestra **el orden en el que se llaman**
y, sobre todo, la diferencia entre el primer turno y los siguientes.

```mermaid
sequenceDiagram
    autonumber
    actor U as Usuario
    participant A as ask
    participant H as sessions
    participant C as contextualize_chain
    participant R as retriever
    participant Q as qa_chain

    Note over U,Q: Turno 1 · la memoria está vacía
    U->>A: ¿Qué platos tiene el recetario?
    A->>H: ¿hay mensajes en esta sesión?
    H-->>A: lista vacía
    Note right of A: no hay nada que reformular,<br/>se usa la pregunta tal cual
    A->>R: invoke(pregunta)
    R-->>A: 4 chunks relevantes
    A->>Q: contexto + mensajes
    Q-->>A: respuesta
    A->>H: guarda Human + AI en sessions[session_id]
    A-->>U: respuesta con emojis y cita

    Note over U,Q: Turno 2 · ahora sí hay historial
    U->>A: ¿y cuántas calorías tienen?
    A->>H: ¿hay mensajes en esta sesión?
    H-->>A: 2 mensajes
    A->>C: historial + pregunta ambigua
    C-->>A: ¿Cuántas calorías tienen los platos del recetario?
    A->>R: invoke(pregunta reformulada)
    R-->>A: 4 chunks relevantes
    A->>Q: contexto + mensajes
    Q-->>A: respuesta
    A->>H: guarda Human + AI en sessions[session_id]
    A-->>U: respuesta
```

Dos cosas que conviene notar:

- **El turno 2 cuesta dos llamadas al modelo**, no una: una para reformular y otra para responder.
  Ese es el precio de la memoria conversacional, y por eso el primer turno se salta el paso.
- **El historial guarda la pregunta original**, no la reformulada. La reformulación es un
  artefacto interno para buscar mejor; si la guardáramos, el historial dejaría de reflejar
  lo que el usuario escribió de verdad.
- **Cada `session_id` tiene su propio historial.** Todo lo demás (vector store, cadenas,
  modelo) se comparte: lo único que cambia entre conversaciones es qué lista de mensajes se usa.


## 1 · Setup

Imports y configuración. Las API keys se leen del `.env` de la raíz del repo;
`find_dotenv()` sube por las carpetas hasta encontrarlo, así que funciona
aunque el notebook esté en `notebooks/`.


In [1]:
import os
from collections import defaultdict
from pathlib import Path

import requests
from dotenv import find_dotenv, load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain_chroma import Chroma
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


/var/folders/ym/1598v2z14n38j4znw7slpmpc0000gn/T/ipykernel_30192/13042150.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader


In [2]:
# find_dotenv sube por las carpetas hasta encontrar el .env de la raíz
load_dotenv(find_dotenv())

# Api keys
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

# embedding model
EMBEDDINGS = OpenAIEmbeddings(model="text-embedding-3-small")

# documents path
DATA_DIR = Path("data/pdfs")
DATA_DIR.mkdir(parents=True, exist_ok=True)


## 2 · Los documentos

Descargamos los PDFs una sola vez a `data/pdfs/` y a partir de ahí trabajamos en local:
es más rápido, funciona sin conexión y evita depender de que las URLs sigan vivas.

> El CDN devuelve **403** a peticiones sin `User-Agent` de navegador, de ahí la cabecera.


In [3]:
# Descarga única de los PDFs a local. El CDN de Platzi devuelve 403
# a peticiones sin User-Agent de navegador.
HEADERS = {"User-Agent": "Mozilla/5.0"}

urls = {
    "informe_analisis_ventas_tienda_abc": "https://static.platzi.com/media/public/uploads/informe_analisis_ventas_tienda_abc_78f886d7-831a-496c-ac41-082a322ea23c.pdf",
    "recetario_cocina_saludable": "https://static.platzi.com/media/public/uploads/recetario_cocina_saludable_e1e8fca6-168b-4c9c-bf47-74e8b97ea02f.pdf",
    "guia_viaje_descubre_paris": "https://static.platzi.com/media/public/uploads/guia_viaje_descubre_paris_78441010-4be3-4f9a-80b3-7093a3acabf7.pdf",
    "investigacion_ia_medicina": "https://static.platzi.com/media/public/uploads/investigacion_ia_medicina_a3889b2c-cb97-4b29-ac95-bd9c9d49c6b7.pdf",
}

for name, url in urls.items():
    dest = DATA_DIR / f"{name}.pdf"
    if dest.exists():
        print(f"Ya existe: {dest.name}")
        continue
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    dest.write_bytes(r.content)
    print(f"Descargado: {dest.name} ({len(r.content)} bytes)")


Ya existe: informe_analisis_ventas_tienda_abc.pdf
Ya existe: recetario_cocina_saludable.pdf
Ya existe: guia_viaje_descubre_paris.pdf
Ya existe: investigacion_ia_medicina.pdf


### Cargarlos con `DirectoryLoader`

`DirectoryLoader` recorre la carpeta y aplica un loader a cada archivo que haga match con el `glob`.
Pasamos `loader_cls=PyPDFLoader` porque por defecto usaría `UnstructuredFileLoader`,
que exige dependencias de sistema extra.

Cada página se convierte en un `Document` con `page_content` y `metadata`
(`source` y `page`), que es lo que después nos permite **citar la fuente**.


In [4]:
# loader_cls=PyPDFLoader: por defecto DirectoryLoader usa Unstructured,
# que exige dependencias extra.
loader = DirectoryLoader(
    str(DATA_DIR),
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True,
)
docs = loader.load()

  0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:00<00:00, 110.66it/s]

In [5]:
docs

[Document(metadata={'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': 'PyPDF', 'creationdate': 'D:20240912034157', 'source': 'data/pdfs/informe_analisis_ventas_tienda_abc.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Informe de Análisis de Ventas - Tienda ABC\nEste informe presenta un análisis detallado de las ventas realizadas en la Tienda ABC durante el\núltimo trimestre. El objetivo de este informe es identificar tendencias, analizar el rendimiento de los\nproductos, y proponer recomendaciones para mejorar las ventas en el próximo período.\n1. Rendimiento General de Ventas\nEn el último trimestre, las ventas totales alcanzaron un valor de $150,000, con un incremento del\n15% respecto al trimestre anterior. El número total de transacciones fue de 2,500, lo que representa\nun promedio de 833 transacciones por mes.\n2. Análisis por Categoría de Producto\nLas siguientes categorías de productos se analizaron según su rendimiento en ventas:\n- Rop

In [6]:
docs[0]

Document(metadata={'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': 'PyPDF', 'creationdate': 'D:20240912034157', 'source': 'data/pdfs/informe_analisis_ventas_tienda_abc.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Informe de Análisis de Ventas - Tienda ABC\nEste informe presenta un análisis detallado de las ventas realizadas en la Tienda ABC durante el\núltimo trimestre. El objetivo de este informe es identificar tendencias, analizar el rendimiento de los\nproductos, y proponer recomendaciones para mejorar las ventas en el próximo período.\n1. Rendimiento General de Ventas\nEn el último trimestre, las ventas totales alcanzaron un valor de $150,000, con un incremento del\n15% respecto al trimestre anterior. El número total de transacciones fue de 2,500, lo que representa\nun promedio de 833 transacciones por mes.\n2. Análisis por Categoría de Producto\nLas siguientes categorías de productos se analizaron según su rendimiento en ventas:\n- Ropa

In [7]:
for metadata in docs:
    print("--"*10)
    print(metadata)

--------------------
page_content='Informe de Análisis de Ventas - Tienda ABC
Este informe presenta un análisis detallado de las ventas realizadas en la Tienda ABC durante el
último trimestre. El objetivo de este informe es identificar tendencias, analizar el rendimiento de los
productos, y proponer recomendaciones para mejorar las ventas en el próximo período.
1. Rendimiento General de Ventas
En el último trimestre, las ventas totales alcanzaron un valor de $150,000, con un incremento del
15% respecto al trimestre anterior. El número total de transacciones fue de 2,500, lo que representa
un promedio de 833 transacciones por mes.
2. Análisis por Categoría de Producto
Las siguientes categorías de productos se analizaron según su rendimiento en ventas:
- Ropa: Representa el 40% de las ventas totales, con un valor de $60,000.
- Calzado: Contribuye con el 30% de las ventas, alcanzando $45,000.
- Accesorios: Generaron $30,000, representando el 20% de las ventas.
- Otros: El 10% restante, eq

## 3 · Trocear en chunks

Una página entera es demasiado grande para un embedding útil: el vector queda "promediado"
y pierde precisión. La troceamos en fragmentos de ~800 caracteres.

El `chunk_overlap=100` hace que los trozos se solapen, para que una frase que cae
justo en el corte no se pierda entre dos chunks.


In [8]:
# 100 caracteres era demasiado poco: partía las frases a la mitad.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
)

chunks = text_splitter.split_documents(docs)
print(len(docs), "páginas →", len(chunks), "chunks")


8 páginas → 13 chunks


In [9]:
chunks

[Document(metadata={'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': 'PyPDF', 'creationdate': 'D:20240912034157', 'source': 'data/pdfs/informe_analisis_ventas_tienda_abc.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Informe de Análisis de Ventas - Tienda ABC\nEste informe presenta un análisis detallado de las ventas realizadas en la Tienda ABC durante el\núltimo trimestre. El objetivo de este informe es identificar tendencias, analizar el rendimiento de los\nproductos, y proponer recomendaciones para mejorar las ventas en el próximo período.\n1. Rendimiento General de Ventas\nEn el último trimestre, las ventas totales alcanzaron un valor de $150,000, con un incremento del\n15% respecto al trimestre anterior. El número total de transacciones fue de 2,500, lo que representa\nun promedio de 833 transacciones por mes.\n2. Análisis por Categoría de Producto\nLas siguientes categorías de productos se analizaron según su rendimiento en ventas:\n- Rop

In [10]:
print(chunks[0])

page_content='Informe de Análisis de Ventas - Tienda ABC
Este informe presenta un análisis detallado de las ventas realizadas en la Tienda ABC durante el
último trimestre. El objetivo de este informe es identificar tendencias, analizar el rendimiento de los
productos, y proponer recomendaciones para mejorar las ventas en el próximo período.
1. Rendimiento General de Ventas
En el último trimestre, las ventas totales alcanzaron un valor de $150,000, con un incremento del
15% respecto al trimestre anterior. El número total de transacciones fue de 2,500, lo que representa
un promedio de 833 transacciones por mes.
2. Análisis por Categoría de Producto
Las siguientes categorías de productos se analizaron según su rendimiento en ventas:
- Ropa: Representa el 40% de las ventas totales, con un valor de $60,000.' metadata={'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': 'PyPDF', 'creationdate': 'D:20240912034157', 'source': 'data/pdfs/informe_analisis_ventas_tienda_abc.pdf',

## 4 · Vector store y retriever

`Chroma.from_documents()` hace dos cosas de una vez: calcula el embedding de cada chunk
y los guarda en la base vectorial.

El `retriever` es la interfaz de consulta: recibe texto, lo convierte a embedding y
devuelve los `k` chunks más cercanos.


In [11]:
# from_documents es classmethod: va sobre la clase, no sobre una instancia.
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=EMBEDDINGS,
)


In [12]:
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

## 5 · Prompts y modelo

Necesitamos **dos** prompts distintos, y esta es la parte conceptualmente importante:

| Prompt | Para qué sirve | Qué devuelve |
|---|---|---|
| `contextualize_prompt` | convertir *"¿y cuántas calorías tienen?"* en una pregunta autónoma | otra **pregunta** |
| `qa_prompt` | responder usando los chunks recuperados | la **respuesta** final |

El primero explícitamente **no** responde: su único trabajo es reescribir.
`MessagesPlaceholder("messages")` es el hueco donde se inyecta el historial en `qa_prompt`.

En `contextualize_prompt` el historial va **como texto** dentro de un único mensaje.
Si se lo pasáramos como conversación, Claude vería una pregunta del usuario al final
y la **contestaría** (inventando datos) en lugar de reformularla, aunque el system prompt diga que no.


In [13]:
# 1) Reformula la pregunta para que se entienda sin el historial.
#    Es el equivalente a create_history_aware_retriever.
contextualize_system_prompt = (
    "Dado el historial de chat y la última pregunta del usuario, reformúlala "
    "como una pregunta autónoma que se entienda sin el historial. "
    "NO respondas la pregunta, solo reformúlala si hace falta."
)
contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_system_prompt),
    ("human",
     "<historial>\n{historial}\n</historial>\n\n"
     "<pregunta>{pregunta}</pregunta>\n\n"
     "Devuelve SOLO la pregunta reformulada, sin responderla ni añadir nada más."),
])

# 2) Responde usando los documentos recuperados.
system_prompt = (
    "Eres un asistente que responde sobre múltiples PDFs. Incluye emojis en cada "
    "respuesta y cita el documento del que sacas la información. "
    "Si la respuesta no está en el contexto, dilo.\n\nContexto:\n{contexto}"
)
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("messages"),
])


In [14]:
# Anthropic no ofrece embeddings: el vector store sigue con OpenAI,
# solo el modelo de chat pasa a Claude.
llm = ChatAnthropic(
    model="claude-haiku-4-5",
    temperature=0,
    max_tokens=400,  # 150 cortaba las respuestas a media frase
)

## 6 · La cadena conversacional

Aquí se junta todo. La **memoria es un diccionario de listas**: una lista de mensajes
por cada `session_id`. No hace falta ninguna abstracción especial — esto es literalmente
lo que hacen por dentro las clases de memoria de LangChain.

El `session_id` es lo que permite atender varias conversaciones a la vez sin que se
mezclen: dos usuarios distintos, o el mismo usuario en dos pestañas, cada uno con su hilo.

Las cadenas se componen con el operador `|` (LCEL): `prompt | modelo | parser`,
que se lee como "el prompt alimenta al modelo, y su salida al parser".


In [15]:
# La memoria: un historial independiente por sesión.
# defaultdict(list) crea la lista vacía la primera vez que se usa un session_id nuevo.
sessions: dict[str, list] = defaultdict(list)


def format_docs(docs):
    """Junta los chunks recuperados en un solo string citable."""
    return "\n\n".join(
        f"[{Path(d.metadata['source']).name} p.{d.metadata.get('page', 0) + 1}]\n{d.page_content}"
        for d in docs
    )


def format_history(mensajes):
    """Convierte el historial en texto plano para el prompt de reformulación."""
    return "\n".join(
        f"{'Usuario' if m.type == 'human' else 'Asistente'}: {m.content}"
        for m in mensajes
    )


# Las dos cadenas LCEL: prompt | modelo | parser
contextualize_chain = contextualize_prompt | llm | StrOutputParser()
qa_chain = qa_prompt | llm | StrOutputParser()


In [16]:
def ask(pregunta: str, session_id: str = "default", verbose: bool = True):
    """Un turno completo: reformular -> buscar -> responder -> guardar en su sesión."""
    chat_history = sessions[session_id]
    mensajes = chat_history + [HumanMessage(pregunta)]

    # 1) Sin historial no hay nada que reformular.
    if chat_history:
        query = contextualize_chain.invoke({
            "historial": format_history(chat_history),
            "pregunta": pregunta,
        })
    else:
        query = pregunta

    # 2) Buscar en Chroma con la pregunta autónoma.
    docs = retriever.invoke(query)

    # 3) Responder con ese contexto.
    respuesta = qa_chain.invoke({
        "contexto": format_docs(docs),
        "messages": mensajes,
    })

    # 4) Guardar el turno en el historial de ESTA sesión.
    chat_history.extend([HumanMessage(pregunta), AIMessage(respuesta)])

    if verbose:
        print(f"[{session_id}] 👤", pregunta)
        print(f"[{session_id}] 🔎 buscó:", query)
        print(f"[{session_id}] 🤖", respuesta)
        print("-" * 60)
    return respuesta


def show_history(session_id: str = "default"):
    """Imprime el historial de una sesión."""
    historial = sessions[session_id]
    print(f"sesión '{session_id}': {len(historial)} mensajes\n")
    for m in historial:
        print(f"  {m.type:10} | {m.content[:65]}")


def reset_session(session_id: str = "default"):
    """Vacía el historial de una sesión, sin tocar las demás."""
    sessions[session_id].clear()


## 7 · Probarlo

Dos cosas que comprobamos aquí:

1. **La memoria funciona**: el segundo `ask()` de la sesión `cocina` pregunta
   *"¿y cuántas calorías tienen?"* sin mencionar el recetario. Mira la línea `🔎 buscó:`
   para ver la reformulación en acción.
2. **Las sesiones están aisladas**: la sesión `viajes` pregunta por París y no sabe
   nada de recetas, aunque comparte el mismo vector store y las mismas cadenas.


In [17]:
ask("¿Qué platos tiene el recetario?", session_id="cocina")
ask("¿y cuántas calorías tienen?", session_id="cocina")   # depende del historial


[cocina] 👤 ¿Qué platos tiene el recetario?
[cocina] 🔎 buscó: ¿Qué platos tiene el recetario?
[cocina] 🤖 🍽️ Según el **recetario_cocina_saludable.pdf**, el recetario incluye los siguientes platos:

1. **Desayuno - Avena con Frutas** 🥣: Avena cocida con leche de almendras, miel y frutas frescas (fresas, plátano, arándanos).

2. **Almuerzo - Ensalada de Quinoa con Aguacate** 🥗: Una ensalada nutritiva con quinoa, aguacate, tomate, espinacas y aliño de limón.

3. **Cena - Salmón al Horno con Vegetales** 🐟: Filetes de salmón horneados acompañados de brócoli y zanahorias, condimentados con ajo, limón, sal y pimienta.

¡Todas son opciones saludables y deliciosas! 😋
------------------------------------------------------------


[cocina] 👤 ¿y cuántas calorías tienen?
[cocina] 🔎 buscó: ¿Cuántas calorías tienen los platos del recetario de cocina saludable (Avena con Frutas, Ensalada de Quinoa con Aguacate y Salmón al Horno con Vegetales)?
[cocina] 🤖 😅 La información sobre calorías **no está disponible en el contexto** que tengo. El **recetario_cocina_saludable.pdf** proporciona los ingredientes e instrucciones de preparación, pero no incluye datos nutricionales ni información calórica de las recetas.

Si necesitas conocer las calorías de estos platos, te recomendaría consultar una tabla nutricional o usar una aplicación de cálculo de calorías basándote en los ingredientes listados. 📊
------------------------------------------------------------


'😅 La información sobre calorías **no está disponible en el contexto** que tengo. El **recetario_cocina_saludable.pdf** proporciona los ingredientes e instrucciones de preparación, pero no incluye datos nutricionales ni información calórica de las recetas.\n\nSi necesitas conocer las calorías de estos platos, te recomendaría consultar una tabla nutricional o usar una aplicación de cálculo de calorías basándote en los ingredientes listados. 📊'

In [18]:
# Otra sesión en paralelo: historial propio, no se entera de lo anterior.
ask("¿Qué se puede visitar en París?", session_id="viajes")


[viajes] 👤 ¿Qué se puede visitar en París?
[viajes] 🔎 buscó: ¿Qué se puede visitar en París?
[viajes] 🤖 # 🗼 Lugares para Visitar en París

¡París tiene muchísimos lugares increíbles! Según la **[guia_viaje_descubre_paris.pdf p.1]**, aquí están los imprescindibles:

## Monumentos Principales 🏛️
- **Torre Eiffel**: El símbolo icónico de París y una de las estructuras más reconocidas del mundo. ¡No te pierdas la vista panorámica desde la cima!
- **Museo del Louvre**: El museo más grande del mundo, hogar de obras maestras como la Mona Lisa y la Venus de Milo 🎨
- **Catedral de Notre-Dame**: Una catedral gótica que impresiona por su arquitectura e historia ⛪
- **Campos Elíseos y Arco del Triunfo**: Pasea por esta avenida famosa y sube al Arco para disfrutar de una vista espectacular de la ciudad 👑

## Experiencias Adicionales 🌟
Además, puedes disfrutar de la gastronomía local, explorar la ciudad en bicicleta o hacer un paseo en barco por el río Sena para ver París desde una perspectiva única

'# 🗼 Lugares para Visitar en París\n\n¡París tiene muchísimos lugares increíbles! Según la **[guia_viaje_descubre_paris.pdf p.1]**, aquí están los imprescindibles:\n\n## Monumentos Principales 🏛️\n- **Torre Eiffel**: El símbolo icónico de París y una de las estructuras más reconocidas del mundo. ¡No te pierdas la vista panorámica desde la cima!\n- **Museo del Louvre**: El museo más grande del mundo, hogar de obras maestras como la Mona Lisa y la Venus de Milo 🎨\n- **Catedral de Notre-Dame**: Una catedral gótica que impresiona por su arquitectura e historia ⛪\n- **Campos Elíseos y Arco del Triunfo**: Pasea por esta avenida famosa y sube al Arco para disfrutar de una vista espectacular de la ciudad 👑\n\n## Experiencias Adicionales 🌟\nAdemás, puedes disfrutar de la gastronomía local, explorar la ciudad en bicicleta o hacer un paseo en barco por el río Sena para ver París desde una perspectiva única 🚴\u200d♂️🛥️\n\n¿Te gustaría saber más sobre transporte, alojamiento o gastronomía en París?

In [19]:
show_history("cocina")
print()
show_history("viajes")

print("\nsesiones activas:", dict((k, len(v)) for k, v in sessions.items()))


sesión 'cocina': 4 mensajes

  human      | ¿Qué platos tiene el recetario?
  ai         | 🍽️ Según el **recetario_cocina_saludable.pdf**, el recetario incl
  human      | ¿y cuántas calorías tienen?
  ai         | 😅 La información sobre calorías **no está disponible en el contex

sesión 'viajes': 2 mensajes

  human      | ¿Qué se puede visitar en París?
  ai         | # 🗼 Lugares para Visitar en París

¡París tiene muchísimos lugare

sesiones activas: {'cocina': 4, 'viajes': 2}
